Tugas 4 No. 2: Pembobotan (Term Weighting)
Topik:Bag-of-Words, TF, IDF, dan TF-IDF

Nama:[Mughni Madya Maylafaisya.M]
NIM: [240210502029]

## 1. Import Library

In [4]:
import re
import math
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print("Library berhasil diimport.")

Library berhasil diimport.


## 2. Preprocessing Sederhana
Case folding + cleaning + tokenisasi + stopwords removal.

In [5]:
stopwords = set(StopWordRemoverFactory().get_stop_words())

def preprocess_simple(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords]
    return tokens

## 3. Dataset (4 Dokumen Pendek)

In [6]:
documents = [
    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak.",
    "Jaringan komputer menghubungkan komputer untuk berbagi data.",
    "Kecerdasan buatan mempelajari cara komputer belajar dari data.",
    "Sistem temu kembali informasi mencari dokumen relevan berdasarkan query."
]

tokens_docs = [preprocess_simple(doc) for doc in documents]
doc_names = [f"D{i+1}" for i in range(len(documents))]

for i, tokens in enumerate(tokens_docs, 1):
    print(f"D{i}: {tokens}")

D1: ['sistem', 'komputer', 'terdiri', 'perangkat', 'keras', 'perangkat', 'lunak']
D2: ['jaringan', 'komputer', 'menghubungkan', 'komputer', 'berbagi', 'data']
D3: ['kecerdasan', 'buatan', 'mempelajari', 'cara', 'komputer', 'belajar', 'data']
D4: ['sistem', 'temu', 'informasi', 'mencari', 'dokumen', 'relevan', 'berdasarkan', 'query']


## 4. Bag-of-Words (Raw Count)

In [7]:
vocab = sorted(set(term for doc in tokens_docs for term in doc))

tf_df = pd.DataFrame(0, index=vocab, columns=doc_names)
for term in vocab:
    for j, doc_tokens in enumerate(tokens_docs):
        tf_df.loc[term, doc_names[j]] = doc_tokens.count(term)

print("BAG-OF-WORDS (RAW COUNT / TF):")
tf_df

BAG-OF-WORDS (RAW COUNT / TF):


,D1,D2,D3,D4
belajar,0,0,1,0
berbagi,0,1,0,0
berdasarkan,0,0,0,1
buatan,0,0,1,0
cara,0,0,1,0
data,0,1,1,0
dokumen,0,0,0,1
informasi,0,0,0,1
jaringan,0,1,0,0
kecerdasan,0,0,1,0


## 5. Perhitungan Manual TF-IDF

Rumus:
- **TF(t,d)** = jumlah kemunculan term t pada dokumen d
- **df(t)** = jumlah dokumen yang mengandung term t
- **IDF(t)** = ln(N / df) + 1  *(setara scikit-learn dengan `smooth_idf=False`)*
- **TF-IDF** = TF × IDF

In [8]:
N = len(documents)
df = (tf_df > 0).sum(axis=1)
idf = df.apply(lambda x: math.log(N / x) + 1)

idf_df = pd.DataFrame({
    'df': df,
    'idf_manual': idf
})

print(f"N = {N}")
print("\nDOCUMENT FREQUENCY (df) DAN IDF MANUAL:")
idf_df

N = 4

DOCUMENT FREQUENCY (df) DAN IDF MANUAL:


,df,idf_manual
belajar,1,2.3863
berbagi,1,2.3863
berdasarkan,1,2.3863
buatan,1,2.3863
cara,1,2.3863
data,2,1.6931
dokumen,1,2.3863
informasi,1,2.3863
jaringan,1,2.3863
kecerdasan,1,2.3863


In [9]:
tfidf_manual = tf_df.multiply(idf, axis=0)
print("TF-IDF MANUAL (TF × IDF):")
tfidf_manual

TF-IDF MANUAL (TF × IDF):


,D1,D2,D3,D4
belajar,0.0000,0.0000,2.3863,0.0000
berbagi,0.0000,2.3863,0.0000,0.0000
berdasarkan,0.0000,0.0000,0.0000,2.3863
buatan,0.0000,0.0000,2.3863,0.0000
cara,0.0000,0.0000,2.3863,0.0000
data,0.0000,1.6931,1.6931,0.0000
dokumen,0.0000,0.0000,0.0000,2.3863
informasi,0.0000,0.0000,0.0000,2.3863
jaringan,0.0000,2.3863,0.0000,0.0000
kecerdasan,0.0000,0.0000,2.3863,0.0000


## 6. TF-IDF dengan scikit-learn (Setara Manual)

Agar hasil sama dengan manual, gunakan `norm=None` dan `smooth_idf=False`.

In [10]:
vectorizer = TfidfVectorizer(
    tokenizer=preprocess_simple,
    lowercase=False,
    norm=None,
    smooth_idf=False
)
X = vectorizer.fit_transform(documents)

sk_tfidf = pd.DataFrame(
    X.toarray().T,
    index=vectorizer.get_feature_names_out(),
    columns=doc_names
)

print("TF-IDF SCIKIT-LEARN (norm=None, smooth_idf=False):")
sk_tfidf

c:\Users\THINKPAD T14S\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TF-IDF SCIKIT-LEARN (norm=None, smooth_idf=False):


,D1,D2,D3,D4
belajar,0.0000,0.0000,2.3863,0.0000
berbagi,0.0000,2.3863,0.0000,0.0000
berdasarkan,0.0000,0.0000,0.0000,2.3863
buatan,0.0000,0.0000,2.3863,0.0000
cara,0.0000,0.0000,2.3863,0.0000
data,0.0000,1.6931,1.6931,0.0000
dokumen,0.0000,0.0000,0.0000,2.3863
informasi,0.0000,0.0000,0.0000,2.3863
jaringan,0.0000,2.3863,0.0000,0.0000
kecerdasan,0.0000,0.0000,2.3863,0.0000


## 7. Perbandingan Manual vs scikit-learn

In [11]:
manual_aligned = tfidf_manual.reindex(sk_tfidf.index).fillna(0)
diff = (manual_aligned - sk_tfidf).abs().max().max()
print(f"Selisih maksimum manual vs scikit-learn: {diff:.10f}")
print("Kesimpulan:", "IDENTIK" if diff < 1e-9 else "BERBEDA")

Selisih maksimum manual vs scikit-learn: 0.0000000000
Kesimpulan: IDENTIK


## 8. TF-IDF scikit-learn (Default)

Default scikit-learn menggunakan `smooth_idf=True` dan normalisasi L2.

In [12]:
vectorizer_default = TfidfVectorizer(
    tokenizer=preprocess_simple,
    lowercase=False
)
X_default = vectorizer_default.fit_transform(documents)

sk_default = pd.DataFrame(
    X_default.toarray().T,
    index=vectorizer_default.get_feature_names_out(),
    columns=doc_names
)

print("TF-IDF SCIKIT-LEARN DEFAULT (smooth_idf=True, norm='l2'):")
sk_default

TF-IDF SCIKIT-LEARN DEFAULT (smooth_idf=True, norm='l2'):


c:\Users\THINKPAD T14S\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,D1,D2,D3,D4
belajar,0.0000,0.0000,0.4073,0.0000
berbagi,0.0000,0.4364,0.0000,0.0000
berdasarkan,0.0000,0.0000,0.0000,0.3622
buatan,0.0000,0.0000,0.4073,0.0000
cara,0.0000,0.0000,0.4073,0.0000
data,0.0000,0.3441,0.3211,0.0000
dokumen,0.0000,0.0000,0.0000,0.3622
informasi,0.0000,0.0000,0.0000,0.3622
jaringan,0.0000,0.4364,0.0000,0.0000
kecerdasan,0.0000,0.0000,0.4073,0.0000


## 9. Term dengan Bobot TF-IDF Tertinggi per Dokumen

In [13]:
print("TERM DENGAN TF-IDF TERTINGGI PER DOKUMEN (MANUAL):")
print("=" * 60)
for col in tfidf_manual.columns:
    term = tfidf_manual[col].idxmax()
    val = tfidf_manual[col].max()
    print(f"{col}: '{term}' dengan bobot = {val:.4f}")

TERM DENGAN TF-IDF TERTINGGI PER DOKUMEN (MANUAL):
D1: 'perangkat' dengan bobot = 4.7726
D2: 'komputer' dengan bobot = 2.5754
D3: 'belajar' dengan bobot = 2.3863
D4: 'berdasarkan' dengan bobot = 2.3863


10. Analisis Singkat

**Term mana yang memiliki bobot TF-IDF tertinggi pada masing-masing dokumen? Mengapa term tersebut penting?**

Term dengan bobot TF-IDF tertinggi pada tiap dokumen adalah term yang **sering muncul di dalam dokumen tersebut (TF tinggi)** tetapi **jarang muncul di dokumen lain (DF rendah, sehingga IDF tinggi)**. Contohnya:
- **D1**: "keras" / "lunak" → khas tentang perangkat komputer.
- **D2**: "jaringan" / "bagi" → khas tentang jaringan komputer.
- **D3**: "kecerdasan" / "belajar" → khas tentang AI.
- **D4**: "temu" / "kembali" / "relevan" → khas tentang sistem temu kembali informasi.

Term-term tersebut menjadi **ciri pembeda (discriminative feature)** antar dokumen. Dalam sistem IR, term dengan bobot TF-IDF tinggi paling berkontribusi saat menghitung **cosine similarity** antara query dan dokumen, sehingga dokumen yang relevan lebih mudah teridentifikasi. Sebaliknya, term yang muncul di semua dokumen (misalnya "komputer" atau "data") memiliki IDF rendah dan bobot kecil, karena tidak cukup membedakan dokumen satu dengan yang lain.